# Part 5: Data Analysis & Exploration

Now that you have generated datasets, let's explore the data and perform some basic analyses.

---

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
dataset_dir = Path('artifacts/datasets')

print("✓ Setup complete")

---

## 1. Load Dataset

Choose a dataset to explore:

In [ ]:
# List available datasets
datasets = list(dataset_dir.glob('*.csv'))

print("Available datasets:\n")
for i, ds in enumerate(sorted(datasets), 1):
    size_mb = ds.stat().st_size / 1024 / 1024
    print(f"{i:2d}. {ds.name:<50s} ({size_mb:.1f} MB)")

In [ ]:
# Load a dataset
# EDIT THIS: Change to your dataset filename
dataset_name = 'my_w6_mortality_dataset.csv'

df = pd.read_csv(dataset_dir / dataset_name)

print(f"Dataset: {dataset_name}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

---

## 2. Data Overview

In [ ]:
# Column types
print("Column Types:\n")
print(df.dtypes.value_counts())

print("\nFirst 5 rows:")
df.head()

In [ ]:
# Missing data analysis
missing = df.isnull().sum()
missing_pct = 100 * missing / len(df)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing': missing.values,
    'Percent': missing_pct.values
}).sort_values('Percent', ascending=False)

print("Missing Data (top 15):\n")
print(missing_df.head(15).to_string(index=False))

In [ ]:
# Visualize missing data
top_missing = missing_df.head(20)

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_missing)), top_missing['Percent'])
plt.yticks(range(len(top_missing)), top_missing['Column'])
plt.xlabel('Missing (%)')
plt.title('Top 20 Variables by Missing Data')
plt.tight_layout()
plt.show()

---

## 3. Demographics Analysis

In [ ]:
# Age distribution
if 'age_at_ed' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df['age_at_ed'].dropna(), bins=30, edgecolor='black')
    axes[0].set_xlabel('Age (years)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Age Distribution')
    axes[0].axvline(df['age_at_ed'].median(), color='red', linestyle='--', label=f"Median: {df['age_at_ed'].median():.1f}")
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(df['age_at_ed'].dropna())
    axes[1].set_ylabel('Age (years)')
    axes[1].set_title('Age Distribution (Box Plot)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Age statistics:")
    print(df['age_at_ed'].describe())
else:
    print("age_at_ed column not found")

In [ ]:
# Gender distribution
if 'gender' in df.columns:
    gender_counts = df['gender'].value_counts()
    
    plt.figure(figsize=(8, 6))
    gender_counts.plot(kind='bar', edgecolor='black')
    plt.xlabel('Gender')
    plt.ylabel('Count')
    plt.title('Gender Distribution')
    plt.xticks(rotation=0)
    
    for i, v in enumerate(gender_counts):
        plt.text(i, v + 1000, f"{v:,} ({v/len(df)*100:.1f}%)", ha='center')
    
    plt.tight_layout()
    plt.show()
else:
    print("gender column not found")

---

## 4. Outcome Analysis

In [ ]:
# Find outcome columns
outcome_cols = [col for col in df.columns if col.startswith('y')]

if outcome_cols:
    print(f"Found {len(outcome_cols)} outcome(s):\n")
    
    outcome_stats = []
    for col in outcome_cols:
        count = df[col].sum()
        rate = df[col].mean()
        outcome_stats.append({
            'Outcome': col,
            'Count': int(count),
            'Rate (%)': f"{rate*100:.2f}"
        })
    
    outcome_df = pd.DataFrame(outcome_stats)
    print(outcome_df.to_string(index=False))
    
    # Visualize
    if len(outcome_cols) > 1:
        rates = [df[col].mean() * 100 for col in outcome_cols]
        
        plt.figure(figsize=(10, 6))
        plt.bar(range(len(outcome_cols)), rates, edgecolor='black')
        plt.xticks(range(len(outcome_cols)), outcome_cols, rotation=45, ha='right')
        plt.ylabel('Outcome Rate (%)')
        plt.title('Outcome Rates')
        plt.tight_layout()
        plt.show()
else:
    print("No outcome columns found (columns starting with 'y')")

---

## 5. Feature Analysis

In [ ]:
# Identify numeric features (excluding IDs and outcomes)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['stay_id', 'subject_id', 'hadm_id'] + outcome_cols
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print(f"Found {len(feature_cols)} numeric features")
print(f"\nExample features: {feature_cols[:10]}")

In [ ]:
# Vital signs distribution
vital_signs = ['sbp_mean_6h', 'hr_mean_6h', 'rr_mean_6h', 'spo2_min_6h']
available_vitals = [v for v in vital_signs if v in df.columns]

if available_vitals:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, vital in enumerate(available_vitals[:4]):
        data = df[vital].dropna()
        axes[i].hist(data, bins=30, edgecolor='black')
        axes[i].set_title(f"{vital}\n(n={len(data):,}, mean={data.mean():.1f})")
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("Vital sign columns not found")

In [ ]:
# Lab values distribution
lab_values = ['lactate_first_6h', 'troponin_first_6h', 'creatinine_first_6h']
available_labs = [lab for lab in lab_values if lab in df.columns]

if available_labs:
    fig, axes = plt.subplots(1, len(available_labs), figsize=(14, 5))
    if len(available_labs) == 1:
        axes = [axes]
    
    for i, lab in enumerate(available_labs):
        data = df[lab].dropna()
        axes[i].hist(data, bins=30, edgecolor='black')
        axes[i].set_title(f"{lab}\n(n={len(data):,})")
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()
else:
    print("Lab value columns not found")

---

## 6. Outcome vs Features Analysis

In [ ]:
# Compare outcomes by age groups
if 'age_at_ed' in df.columns and outcome_cols:
    # Create age groups
    df['age_group'] = pd.cut(df['age_at_ed'], 
                              bins=[0, 30, 50, 65, 80, 120],
                              labels=['18-29', '30-49', '50-64', '65-79', '80+'])
    
    # Calculate outcome rates by age group
    outcome_col = outcome_cols[0]  # Use first outcome
    age_outcome = df.groupby('age_group')[outcome_col].agg(['sum', 'count', 'mean'])
    age_outcome['rate_pct'] = age_outcome['mean'] * 100
    
    print(f"Outcome rates by age group ({outcome_col}):\n")
    print(age_outcome[['count', 'sum', 'rate_pct']].to_string())
    
    # Plot
    plt.figure(figsize=(10, 6))
    age_outcome['rate_pct'].plot(kind='bar', edgecolor='black')
    plt.xlabel('Age Group')
    plt.ylabel('Outcome Rate (%)')
    plt.title(f'{outcome_col} by Age Group')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Feature comparison by outcome
if outcome_cols and 'sbp_mean_6h' in df.columns:
    outcome_col = outcome_cols[0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # SBP by outcome
    df.boxplot(column='sbp_mean_6h', by=outcome_col, ax=axes[0])
    axes[0].set_title('SBP by Outcome')
    axes[0].set_xlabel(outcome_col)
    axes[0].set_ylabel('SBP (mmHg)')
    plt.suptitle('')  # Remove default title
    
    # Heart rate by outcome
    if 'hr_mean_6h' in df.columns:
        df.boxplot(column='hr_mean_6h', by=outcome_col, ax=axes[1])
        axes[1].set_title('Heart Rate by Outcome')
        axes[1].set_xlabel(outcome_col)
        axes[1].set_ylabel('Heart Rate (bpm)')
    
    plt.tight_layout()
    plt.show()

---

## 7. Correlation Analysis

In [ ]:
# Select subset of key features
key_features = [
    'age_at_ed',
    'sbp_mean_6h', 'hr_mean_6h', 'rr_mean_6h', 'spo2_min_6h',
    'lactate_first_6h', 'troponin_first_6h', 'creatinine_first_6h'
]

available_features = [f for f in key_features if f in df.columns]

if len(available_features) >= 3:
    # Correlation matrix
    corr_data = df[available_features].corr()
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                square=True, linewidths=1)
    plt.title('Feature Correlation Matrix')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough features for correlation analysis")

---

## 8. Summary Statistics Report

In [ ]:
print("="*80)
print("DATASET SUMMARY REPORT")
print("="*80)

print(f"\nDataset: {dataset_name}")
print(f"Total records: {len(df):,}")
print(f"Total features: {len(df.columns)}")

if 'age_at_ed' in df.columns:
    print(f"\nAge range: {df['age_at_ed'].min():.0f} - {df['age_at_ed'].max():.0f} years")
    print(f"Mean age: {df['age_at_ed'].mean():.1f} ± {df['age_at_ed'].std():.1f} years")

if 'gender' in df.columns:
    gender_dist = df['gender'].value_counts()
    print(f"\nGender distribution:")
    for gender, count in gender_dist.items():
        print(f"  {gender}: {count:,} ({count/len(df)*100:.1f}%)")

if outcome_cols:
    print(f"\nOutcomes:")
    for col in outcome_cols:
        count = df[col].sum()
        rate = df[col].mean()
        print(f"  {col}: {int(count):,} events ({rate*100:.2f}%)")

print(f"\nMissing data:")
print(f"  Features with >50% missing: {(missing_pct > 50).sum()}")
print(f"  Features with >75% missing: {(missing_pct > 75).sum()}")

print("\n" + "="*80)
print("✓ Analysis complete")
print("="*80)

---

## 🎉 Congratulations!

You've successfully:
1. ✓ Set up the pipeline and database connection
2. ✓ Run the feature extraction pipeline
3. ✓ Generated analysis-ready datasets
4. ✓ Explored and analyzed the data

### Next Steps

- **Machine Learning:** Use your datasets with scikit-learn, XGBoost, etc.
- **Custom Analysis:** Write SQL queries for specific research questions
- **Hybrid Datasets:** Combine multiple windows and outcomes
- **Documentation:** See `docs/COMPLETE_DOCUMENTATION.md` for full reference

---